In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import time
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    when,
    isnan
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# 2. CONFIGURE PYSPARK
# ============================================================

# Install Java and PySpark in Google Colab if needed
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq

!pip install -q pyspark

print("PySpark environment configured.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
PySpark environment configured.


In [3]:
# ============================================================
# 3. CREATE SPARK SESSION
# ============================================================

spark = (
    SparkSession.builder
    .appName("IST3134 US Accident Analysis")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark session created successfully.")
print("Spark version:", spark.version)

Spark session created successfully.
Spark version: 4.0.3


In [4]:
# ============================================================
# 4. SET DATASET PATH
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [5]:
DATASET_PATH = "/content/drive/MyDrive/US_Accidents_March23.csv"

print("Dataset path:")
print(DATASET_PATH)

Dataset path:
/content/drive/MyDrive/US_Accidents_March23.csv


In [6]:
# Check dataset path

if os.path.exists(DATASET_PATH):
    print("Dataset file found.")
else:
    print("ERROR: Dataset file not found.")
    print("Please check DATASET_PATH.")

Dataset file found.


In [7]:
# ============================================================
# 5. LOAD DATASET
# ============================================================

start_time = time.time()

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATASET_PATH)
)

load_time = time.time() - start_time

print("Dataset loaded successfully.")
print("Loading time:", round(load_time, 2), "seconds")

Dataset loaded successfully.
Loading time: 72.46 seconds


In [8]:
# ============================================================
# 6. INSPECT COLUMNS
# ============================================================

print("Total columns:", len(df.columns))
print("\nColumn names:")

for column in df.columns:
    print(column)

Total columns: 46

Column names:
ID
Source
Severity
Start_Time
End_Time
Start_Lat
Start_Lng
End_Lat
End_Lng
Distance(mi)
Description
Street
City
County
State
Zipcode
Country
Timezone
Airport_Code
Weather_Timestamp
Temperature(F)
Wind_Chill(F)
Humidity(%)
Pressure(in)
Visibility(mi)
Wind_Direction
Wind_Speed(mph)
Precipitation(in)
Weather_Condition
Amenity
Bump
Crossing
Give_Way
Junction
No_Exit
Railway
Roundabout
Station
Stop
Traffic_Calming
Traffic_Signal
Turning_Loop
Sunrise_Sunset
Civil_Twilight
Nautical_Twilight
Astronomical_Twilight


In [9]:
# ============================================================
# 7. PRINT SCHEMA
# ============================================================

df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- V

In [10]:
# ============================================================
# 8. COUNT RECORDS
# ============================================================

initial_record_count = df.count()

print("Total records:", initial_record_count)

Total records: 7728394


In [11]:
# ============================================================
# 9. DISPLAY SAMPLE
# ============================================================

df.show(5, truncate=False)

+---+-------+--------+-------------------+-------------------+-----------------+------------------+-------+-------+------------+-------------------------------------------------------------------------------------+-------------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|ID |Source |Severity|Start_Time         |End_Time           |Start_Lat        |Start_Lng         |End_Lat|End_Lng|Distance(mi)|Description                                                                          |Street                   |City        |County    |State|Zipcode   |Country|Timezone  |Airport_Code|Weather_Timestamp  |T

In [13]:
import pyspark.sql.types as T

# ============================================================
# 10. CHECK MISSING VALUES
# ============================================================

missing_values = df.select([
    count(
        when(
            col(c).isNull() |
            (isnan(col(c)) if isinstance(df.schema[c].dataType, (T.DoubleType, T.FloatType)) else False),
            c
        )
    ).alias(c)
    for c in df.columns
])

missing_values.show(truncate=False)

+---+------+--------+----------+--------+---------+---------+-------+-------+------------+-----------+------+----+------+-----+-------+-------+--------+------------+-----------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+----+--------+--------+--------+-------+-------+----------+-------+----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|ID |Source|Severity|Start_Time|End_Time|Start_Lat|Start_Lng|End_Lat|End_Lng|Distance(mi)|Description|Street|City|County|State|Zipcode|Country|Timezone|Airport_Code|Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|Amenity|Bump|Crossing|Give_Way|Junction|No_Exit|Railway|Roundabout|Station|Stop|Traffic_Calming|Traffic_Signal|Turning_Loop|Sunrise_Sunset|Civil_Twilight|Nautical_Twil

In [15]:
# ============================================================
# 11. CHECK IMPORTANT MISSING VALUES
# ============================================================

important_columns = [
    "State",
    "Severity",
    "Weather_Condition"
]

import pyspark.sql.types as T

important_missing = df.select([
    count(
        when(
            col(c).isNull() |
            (isnan(col(c)) if isinstance(df.schema[c].dataType, (T.DoubleType, T.FloatType)) else False),
            c
        )
    ).alias(c)
    for c in important_columns
])

important_missing.show()

+-----+--------+-----------------+
|State|Severity|Weather_Condition|
+-----+--------+-----------------+
|    0|       0|           173459|
+-----+--------+-----------------+



In [16]:
# ============================================================
# 12. REMOVE MISSING VALUES
# ============================================================

before_missing_removal = df.count()

df_clean = df.dropna(
    subset=[
        "State",
        "Severity",
        "Weather_Condition"
    ]
)

after_missing_removal = df_clean.count()

print("Records before removing missing values:",
      before_missing_removal)

print("Records after removing missing values:",
      after_missing_removal)

print("Records removed:",
      before_missing_removal - after_missing_removal)

Records before removing missing values: 7728394
Records after removing missing values: 7554935
Records removed: 173459


In [17]:
# ============================================================
# 13. CHECK DUPLICATES
# ============================================================

before_duplicate_removal = df_clean.count()

distinct_count = df_clean.dropDuplicates().count()

duplicate_count = (
    before_duplicate_removal - distinct_count
)

print("Records before duplicate removal:",
      before_duplicate_removal)

print("Distinct records:",
      distinct_count)

print("Duplicate records detected:",
      duplicate_count)

Records before duplicate removal: 7554935
Distinct records: 7554935
Duplicate records detected: 0


In [18]:
# ============================================================
# 14. REMOVE DUPLICATES
# ============================================================

df_clean = df_clean.dropDuplicates()

after_duplicate_removal = df_clean.count()

print("Records after duplicate removal:",
      after_duplicate_removal)

Records after duplicate removal: 7554935


In [20]:
# ============================================================
# 15. SELECT RELEVANT COLUMNS
# ============================================================

selected_columns = [
    "ID",
    "Severity",
    "Start_Time",
    "State",
    "Weather_Condition",
    "Temperature(F)",
    "Visibility(mi)",
    "Start_Lat",
    "Start_Lng"
]

df_clean = df_clean.select(selected_columns)

print("Relevant columns selected.")

Relevant columns selected.


In [21]:
# ============================================================
# 16. VERIFY SELECTED COLUMNS
# ============================================================

print("Selected columns:")

for column in df_clean.columns:
    print(column)

print("\nTotal selected columns:",
      len(df_clean.columns))

Selected columns:
ID
Severity
Start_Time
State
Weather_Condition
Temperature(F)
Visibility(mi)
Start_Lat
Start_Lng

Total selected columns: 9


In [22]:
# ============================================================
# 17. CONVERT SEVERITY
# ============================================================

df_clean = df_clean.withColumn(
    "Severity",
    col("Severity").cast("integer")
)

print("Severity converted to integer.")

Severity converted to integer.


In [23]:
# ============================================================
# 18. CONVERT START_TIME
# ============================================================

df_clean = df_clean.withColumn(
    "Start_Time",
    col("Start_Time").cast("timestamp")
)

print("Start_Time converted to timestamp.")

Start_Time converted to timestamp.


In [24]:
# ============================================================
# 19. VERIFY DATA TYPES
# ============================================================

df_clean.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- State: string (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Visibility(mi): double (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)



In [25]:
# ============================================================
# 20. FINAL CLEANING VERIFICATION
# ============================================================

final_record_count = df_clean.count()

print("========== FINAL CLEANING VERIFICATION ==========")
print("Original records:", initial_record_count)
print("Final cleaned records:", final_record_count)
print("Records removed:",
      initial_record_count - final_record_count)

print("\nFinal number of columns:",
      len(df_clean.columns))

print("\nFinal columns:")
print(df_clean.columns)

========== FINAL CLEANING VERIFICATION ==========
Original records: 7728394
Final cleaned records: 7554935
Records removed: 173459

Final number of columns: 9

Final columns:
['ID', 'Severity', 'Start_Time', 'State', 'Weather_Condition', 'Temperature(F)', 'Visibility(mi)', 'Start_Lat', 'Start_Lng']


In [27]:
# ============================================================
# 21. DISPLAY CLEANED DATASET
# ============================================================

df_clean.show(10, truncate=False)

+---------+--------+-------------------+-----+----------------------+--------------+--------------+---------+------------------+
|ID       |Severity|Start_Time         |State|Weather_Condition     |Temperature(F)|Visibility(mi)|Start_Lat|Start_Lng         |
+---------+--------+-------------------+-----+----------------------+--------------+--------------+---------+------------------+
|A-3412671|2       |2016-02-09 06:46:32|OH   |Light Snow            |21.0          |1.5           |39.15267 |-84.5395          |
|A-3412699|3       |2016-02-10 08:39:16|OH   |Light Snow            |14.0          |1.5           |39.71548 |-84.22033         |
|A-3412710|2       |2016-02-10 10:47:49|OH   |Light Snow            |17.1          |1.5           |39.93849 |-82.84849         |
|A-3412763|2       |2016-02-11 14:53:15|OH   |Partly Cloudy         |25.0          |10.0          |39.17736 |-84.4873          |
|A-3412815|2       |2016-02-15 05:45:22|OH   |Light Freezing Drizzle|21.9          |5.0          

In [28]:
# ============================================================
# 22. CACHE DATASET
# ============================================================

df_clean.cache()

print("Clean dataset cached.")

Clean dataset cached.


In [29]:
# ============================================================
# 23. CREATE SPARK SQL TEMPORARY VIEW
# ============================================================

df_clean.createOrReplaceTempView("accidents")

print("Spark SQL temporary view 'accidents' created.")

Spark SQL temporary view 'accidents' created.


In [31]:
# ============================================================
# 24. STATE MAP
# ============================================================

state_map = (
    df_clean
    .select("State")
    .rdd
    .map(lambda row: (row["State"], 1))
)

print("State mapper created successfully.")

State mapper created successfully.


In [32]:
# ============================================================
# 25. TEST PYTHON WORKER
# ============================================================

test_result = (
    df_clean
    .select("State")
    .rdd
    .map(lambda row: (row["State"], 1))
    .take(5)
)

print("Python worker test:")
print(test_result)

Python worker test:
[('OH', 1), ('OH', 1), ('OH', 1), ('OH', 1), ('OH', 1)]


In [33]:
# ============================================================
# 26. DISPLAY STATE MAPPER
# ============================================================

print("State Mapper Output:")

for item in state_map.take(10):
    print(item)

State Mapper Output:
('OH', 1)
('OH', 1)
('OH', 1)
('OH', 1)
('OH', 1)
('OH', 1)
('OH', 1)
('IN', 1)
('PA', 1)
('OH', 1)


In [34]:
# ============================================================
# 27. STATE REDUCER
# ============================================================

state_reduced = (
    state_map
    .reduceByKey(lambda a, b: a + b)
)

print("State reducer created successfully.")

State reducer created successfully.


In [35]:
# ============================================================
# 28. SORT STATE
# ============================================================

state_sorted = (
    state_reduced
    .sortBy(lambda x: x[1], ascending=False)
)

print("State results sorted.")

State results sorted.


In [36]:
# ============================================================
# 29. STATE OUTPUT
# ============================================================

print("Top 20 States by Accident Count:")

for state, total in state_sorted.take(20):
    print(state, total)

Top 20 States by Accident Count:
CA 1701655
FL 870292
TX 573638
SC 375402
NY 345234
NC 334204
PA 290596
VA 285338
MN 189010
OR 176447
IL 168031
TN 166816
GA 166529
AZ 164548
MI 161765
LA 148418
NJ 137116
OH 117354
MD 113529
WA 107499


In [37]:
# ============================================================
# 30. STATE DATAFRAME
# ============================================================

state_analysis = (
    state_sorted
    .toDF(["State", "Accident_Count"])
)

state_analysis.show(20, truncate=False)

+-----+--------------+
|State|Accident_Count|
+-----+--------------+
|CA   |1701655       |
|FL   |870292        |
|TX   |573638        |
|SC   |375402        |
|NY   |345234        |
|NC   |334204        |
|PA   |290596        |
|VA   |285338        |
|MN   |189010        |
|OR   |176447        |
|IL   |168031        |
|TN   |166816        |
|GA   |166529        |
|AZ   |164548        |
|MI   |161765        |
|LA   |148418        |
|NJ   |137116        |
|OH   |117354        |
|MD   |113529        |
|WA   |107499        |
+-----+--------------+
only showing top 20 rows


In [38]:
# ============================================================
# 31. SEVERITY MAP
# ============================================================

severity_map = (
    df_clean
    .select("Severity")
    .rdd
    .map(lambda row: (row["Severity"], 1))
)

print("Severity mapper created successfully.")

Severity mapper created successfully.


In [39]:
# ============================================================
# 32. SEVERITY REDUCER
# ============================================================

severity_reduced = (
    severity_map
    .reduceByKey(lambda a, b: a + b)
)

severity_sorted = (
    severity_reduced
    .sortBy(lambda x: x[0], ascending=True)
)

print("Severity reducer created successfully.")

Severity reducer created successfully.


In [40]:
# ============================================================
# 33. SEVERITY OUTPUT
# ============================================================

print("Severity Results:")

for severity, total in severity_sorted.collect():
    print("Severity:", severity,
          "| Total:", total)

Severity Results:
Severity: 1 | Total: 66412
Severity: 2 | Total: 6022200
Severity: 3 | Total: 1269042
Severity: 4 | Total: 197281


In [41]:
# ============================================================
# 34. SEVERITY DATAFRAME
# ============================================================

severity_analysis = (
    severity_sorted
    .toDF(["Severity", "Accident_Count"])
)

severity_analysis.show()

+--------+--------------+
|Severity|Accident_Count|
+--------+--------------+
|       1|         66412|
|       2|       6022200|
|       3|       1269042|
|       4|        197281|
+--------+--------------+



In [42]:
# ============================================================
# 35. WEATHER MAP
# ============================================================

weather_map = (
    df_clean
    .select("Weather_Condition")
    .rdd
    .map(lambda row: (row["Weather_Condition"], 1))
)

print("Weather mapper created successfully.")

Weather mapper created successfully.


In [43]:
# ============================================================
# 36. WEATHER REDUCER
# ============================================================

weather_reduced = (
    weather_map
    .reduceByKey(lambda a, b: a + b)
)

weather_sorted = (
    weather_reduced
    .sortBy(lambda x: x[1], ascending=False)
)

print("Weather reducer created successfully.")

Weather reducer created successfully.


In [44]:
# ============================================================
# 37. WEATHER OUTPUT
# ============================================================

print("Top 20 Weather Conditions:")

for weather, total in weather_sorted.take(20):
    print(weather, total)

Top 20 Weather Conditions:
Fair 2560802
Mostly Cloudy 1016195
Cloudy 817082
Clear 808743
Partly Cloudy 698972
Overcast 382866
Light Rain 352957
Scattered Clouds 204829
Light Snow 128680
Fog 99238
Rain 84331
Haze 76223
Fair / Windy 35671
Heavy Rain 32309
Light Drizzle 22684
Thunder in the Vicinity 17611
Cloudy / Windy 17035
T-Storm 16810
Mostly Cloudy / Windy 16508
Snow 15537


In [45]:
# ============================================================
# 38. WEATHER DATAFRAME
# ============================================================

weather_analysis = (
    weather_sorted
    .toDF(["Weather_Condition", "Accident_Count"])
)

weather_analysis.show(20, truncate=False)

+-----------------------+--------------+
|Weather_Condition      |Accident_Count|
+-----------------------+--------------+
|Fair                   |2560802       |
|Mostly Cloudy          |1016195       |
|Cloudy                 |817082        |
|Clear                  |808743        |
|Partly Cloudy          |698972        |
|Overcast               |382866        |
|Light Rain             |352957        |
|Scattered Clouds       |204829        |
|Light Snow             |128680        |
|Fog                    |99238         |
|Rain                   |84331         |
|Haze                   |76223         |
|Fair / Windy           |35671         |
|Heavy Rain             |32309         |
|Light Drizzle          |22684         |
|Thunder in the Vicinity|17611         |
|Cloudy / Windy         |17035         |
|T-Storm                |16810         |
|Mostly Cloudy / Windy  |16508         |
|Snow                   |15537         |
+-----------------------+--------------+
only showing top

In [46]:
# ============================================================
# 39. SPARK SQL STATE
# ============================================================

sql_state = spark.sql("""
    SELECT
        State,
        COUNT(*) AS Accident_Count
    FROM accidents
    GROUP BY State
    ORDER BY Accident_Count DESC
""")

sql_state.show(20, truncate=False)

+-----+--------------+
|State|Accident_Count|
+-----+--------------+
|CA   |1701655       |
|FL   |870292        |
|TX   |573638        |
|SC   |375402        |
|NY   |345234        |
|NC   |334204        |
|PA   |290596        |
|VA   |285338        |
|MN   |189010        |
|OR   |176447        |
|IL   |168031        |
|TN   |166816        |
|GA   |166529        |
|AZ   |164548        |
|MI   |161765        |
|LA   |148418        |
|NJ   |137116        |
|OH   |117354        |
|MD   |113529        |
|WA   |107499        |
+-----+--------------+
only showing top 20 rows


In [47]:
# ============================================================
# 40. SPARK SQL SEVERITY
# ============================================================

sql_severity = spark.sql("""
    SELECT
        Severity,
        COUNT(*) AS Accident_Count
    FROM accidents
    GROUP BY Severity
    ORDER BY Severity
""")

sql_severity.show()

+--------+--------------+
|Severity|Accident_Count|
+--------+--------------+
|       1|         66412|
|       2|       6022200|
|       3|       1269042|
|       4|        197281|
+--------+--------------+



In [48]:
# ============================================================
# 41. SPARK SQL WEATHER
# ============================================================

sql_weather = spark.sql("""
    SELECT
        Weather_Condition,
        COUNT(*) AS Accident_Count
    FROM accidents
    GROUP BY Weather_Condition
    ORDER BY Accident_Count DESC
""")

sql_weather.show(20, truncate=False)

+-----------------------+--------------+
|Weather_Condition      |Accident_Count|
+-----------------------+--------------+
|Fair                   |2560802       |
|Mostly Cloudy          |1016195       |
|Cloudy                 |817082        |
|Clear                  |808743        |
|Partly Cloudy          |698972        |
|Overcast               |382866        |
|Light Rain             |352957        |
|Scattered Clouds       |204829        |
|Light Snow             |128680        |
|Fog                    |99238         |
|Rain                   |84331         |
|Haze                   |76223         |
|Fair / Windy           |35671         |
|Heavy Rain             |32309         |
|Light Drizzle          |22684         |
|Thunder in the Vicinity|17611         |
|Cloudy / Windy         |17035         |
|T-Storm                |16810         |
|Mostly Cloudy / Windy  |16508         |
|Snow                   |15537         |
+-----------------------+--------------+
only showing top

In [49]:
# ============================================================
# 42. DATAFRAME STATE
# ============================================================

df_state = (
    df_clean
    .groupBy("State")
    .count()
    .withColumnRenamed("count", "Accident_Count")
    .orderBy(
        col("Accident_Count").desc()
    )
)

df_state.show(20, truncate=False)

+-----+--------------+
|State|Accident_Count|
+-----+--------------+
|CA   |1701655       |
|FL   |870292        |
|TX   |573638        |
|SC   |375402        |
|NY   |345234        |
|NC   |334204        |
|PA   |290596        |
|VA   |285338        |
|MN   |189010        |
|OR   |176447        |
|IL   |168031        |
|TN   |166816        |
|GA   |166529        |
|AZ   |164548        |
|MI   |161765        |
|LA   |148418        |
|NJ   |137116        |
|OH   |117354        |
|MD   |113529        |
|WA   |107499        |
+-----+--------------+
only showing top 20 rows


In [50]:
# ============================================================
# 43. DATAFRAME SEVERITY
# ============================================================

df_severity = (
    df_clean
    .groupBy("Severity")
    .count()
    .withColumnRenamed("count", "Accident_Count")
    .orderBy("Severity")
)

df_severity.show()

+--------+--------------+
|Severity|Accident_Count|
+--------+--------------+
|       1|         66412|
|       2|       6022200|
|       3|       1269042|
|       4|        197281|
+--------+--------------+



In [51]:
# ============================================================
# 44. DATAFRAME WEATHER
# ============================================================

df_weather = (
    df_clean
    .groupBy("Weather_Condition")
    .count()
    .withColumnRenamed("count", "Accident_Count")
    .orderBy(
        col("Accident_Count").desc()
    )
)

df_weather.show(20, truncate=False)

+-----------------------+--------------+
|Weather_Condition      |Accident_Count|
+-----------------------+--------------+
|Fair                   |2560802       |
|Mostly Cloudy          |1016195       |
|Cloudy                 |817082        |
|Clear                  |808743        |
|Partly Cloudy          |698972        |
|Overcast               |382866        |
|Light Rain             |352957        |
|Scattered Clouds       |204829        |
|Light Snow             |128680        |
|Fog                    |99238         |
|Rain                   |84331         |
|Haze                   |76223         |
|Fair / Windy           |35671         |
|Heavy Rain             |32309         |
|Light Drizzle          |22684         |
|Thunder in the Vicinity|17611         |
|Cloudy / Windy         |17035         |
|T-Storm                |16810         |
|Mostly Cloudy / Windy  |16508         |
|Snow                   |15537         |
+-----------------------+--------------+
only showing top

In [52]:
# ============================================================
# 45. VERIFY RESULTS
# ============================================================

print("========== RESULT VERIFICATION ==========")

print("\nState MapReduce:")
state_analysis.show(10, truncate=False)

print("\nSeverity MapReduce:")
severity_analysis.show()

print("\nWeather MapReduce:")
weather_analysis.show(10, truncate=False)

print("\nSpark SQL State:")
sql_state.show(10, truncate=False)

print("\nSpark DataFrame State:")
df_state.show(10, truncate=False)

========== RESULT VERIFICATION ==========

State MapReduce:
+-----+--------------+
|State|Accident_Count|
+-----+--------------+
|CA   |1701655       |
|FL   |870292        |
|TX   |573638        |
|SC   |375402        |
|NY   |345234        |
|NC   |334204        |
|PA   |290596        |
|VA   |285338        |
|MN   |189010        |
|OR   |176447        |
+-----+--------------+
only showing top 10 rows

Severity MapReduce:
+--------+--------------+
|Severity|Accident_Count|
+--------+--------------+
|       1|         66412|
|       2|       6022200|
|       3|       1269042|
|       4|        197281|
+--------+--------------+


Weather MapReduce:
+-----------------+--------------+
|Weather_Condition|Accident_Count|
+-----------------+--------------+
|Fair             |2560802       |
|Mostly Cloudy    |1016195       |
|Cloudy           |817082        |
|Clear            |808743        |
|Partly Cloudy    |698972        |
|Overcast         |382866        |
|Light Rain       |352957   

In [53]:
# ============================================================
# 46. SAVE STATE OUTPUT
# ============================================================

OUTPUT_PATH = "/content/drive/MyDrive/IST3134/Output"

os.makedirs(OUTPUT_PATH, exist_ok=True)

state_output_path = os.path.join(
    OUTPUT_PATH,
    "State_Analysis"
)

state_analysis.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(state_output_path)

print("State analysis saved successfully.")
print(state_output_path)

State analysis saved successfully.
/content/drive/MyDrive/IST3134/Output/State_Analysis


In [54]:
# ============================================================
# 47. SAVE SEVERITY OUTPUT
# ============================================================

severity_output_path = os.path.join(
    OUTPUT_PATH,
    "Severity_Analysis"
)

severity_analysis.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(severity_output_path)

print("Severity analysis saved successfully.")
print(severity_output_path)

Severity analysis saved successfully.
/content/drive/MyDrive/IST3134/Output/Severity_Analysis


In [55]:
# ============================================================
# 48. SAVE WEATHER OUTPUT
# ============================================================

weather_output_path = os.path.join(
    OUTPUT_PATH,
    "Weather_Analysis"
)

weather_analysis.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(weather_output_path)

print("Weather analysis saved successfully.")
print(weather_output_path)

Weather analysis saved successfully.
/content/drive/MyDrive/IST3134/Output/Weather_Analysis


In [56]:
# ============================================================
# 49. SAVE CLEANED DATASET
# ============================================================

cleaned_output_path = os.path.join(
    OUTPUT_PATH,
    "Cleaned_US_Accidents"
)

df_clean.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(cleaned_output_path)

print("Cleaned dataset saved successfully.")
print(cleaned_output_path)

Cleaned dataset saved successfully.
/content/drive/MyDrive/IST3134/Output/Cleaned_US_Accidents


In [57]:
# ============================================================
# 50. RECORD SPARK EXECUTION TIME
# ============================================================

start_spark_time = time.time()

# State
timed_state = (
    df_clean
    .select("State")
    .rdd
    .map(lambda row: (row["State"], 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

# Severity
timed_severity = (
    df_clean
    .select("Severity")
    .rdd
    .map(lambda row: (row["Severity"], 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

# Weather
timed_weather = (
    df_clean
    .select("Weather_Condition")
    .rdd
    .map(lambda row: (row["Weather_Condition"], 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

spark_execution_time = time.time() - start_spark_time

print(
    "Spark MapReduce execution time:",
    round(spark_execution_time, 2),
    "seconds"
)

Spark MapReduce execution time: 96.78 seconds


In [58]:
# ============================================================
# 51. FINAL SUMMARY
# ============================================================

print("=" * 60)
print("IST3134 BIG DATA ANALYTICS - FINAL SUMMARY")
print("=" * 60)

print("\nDataset:")
print("US Accidents Dataset")

print("\nOriginal records:")
print(initial_record_count)

print("\nFinal cleaned records:")
print(final_record_count)

print("\nRecords removed:")
print(initial_record_count - final_record_count)

print("\nNumber of selected attributes:")
print(len(df_clean.columns))

print("\nState analysis:")
print(state_analysis.count(), "states")

print("\nSeverity analysis:")
print(severity_analysis.count(), "severity levels")

print("\nWeather analysis:")
print(weather_analysis.count(), "weather conditions")

print("\nSpark MapReduce execution time:")
print(round(spark_execution_time, 2), "seconds")

print("\nOutput directory:")
print(OUTPUT_PATH)

print("\nAll Member 1 Spark processing completed.")
print("=" * 60)

IST3134 BIG DATA ANALYTICS - FINAL SUMMARY

Dataset:
US Accidents Dataset

Original records:
7728394

Final cleaned records:
7554935

Records removed:
173459

Number of selected attributes:
9

State analysis:
49 states

Severity analysis:
4 severity levels

Weather analysis:
144 weather conditions

Spark MapReduce execution time:
96.78 seconds

Output directory:
/content/drive/MyDrive/IST3134/Output

All Member 1 Spark processing completed.


In [59]:
# ============================================================
# 52. STOP SPARK
# ============================================================

spark.stop()

print("Spark session stopped successfully.")

Spark session stopped successfully.
